In [ ]:
import dai
import pandas as pd
import numpy as np


def main(data_source, start_date, end_date):
    """
    Enhanced Multi-Factor Alpha - BigAlpha 2026
    整合: 10档订单簿 + K线形态 + 成交活跃度 + 订单质量 + 财务数据
    """
    # ============================================
    # 1. 获取分钟行情和盘口数据 (全10档)
    # ============================================
    bar_fields = []
    for i in range(1, 11):
        bar_fields.extend([
            f'bid_price{i}', f'ask_price{i}',
            f'bid_volume{i}', f'ask_volume{i}',
            f'bid_num_orders{i}', f'ask_num_orders{i}'
        ])
    
    sql_bar = f"""
    SELECT
    date, instrument,
    open, high, low, price, pre_close,
    volume, amount, num_trades,
    total_bid_volume, total_ask_volume,
    bid_avg_price, ask_avg_price,
    {', '.join(bar_fields)}
    FROM bigalpha_2026_stock_bar1m
    WHERE date >= '{start_date}' AND date <= '{end_date}'
    """
    df = data_source.read(sql_bar)
    
    # ============================================
    # 2. 分钟级特征工程
    # ============================================
    
    # --- 2a. 10档指数衰减权重 ---
    raw_weights = np.array([2.0 ** (-i) for i in range(10)])
    weights = raw_weights / raw_weights.sum()
    
    # --- 2b. OBI (Order Book Imbalance) - 10档加权 ---
    obi_vol = np.zeros(len(df))
    obi_amt = np.zeros(len(df))
    for i in range(1, 11):
        w = weights[i - 1]
        bid_v = df[f'bid_volume{i}'].fillna(0)
        ask_v = df[f'ask_volume{i}'].fillna(0)
        bid_p = df[f'bid_price{i}'].fillna(0)
        ask_p = df[f'ask_price{i}'].fillna(0)
        obi_vol += w * (bid_v - ask_v) / (bid_v + ask_v + 1)
        obi_amt += w * (bid_p * bid_v - ask_p * ask_v) / (
            bid_p * bid_v + ask_p * ask_v + 1
        )
    df['obi_vol'] = obi_vol
    df['obi_amt'] = obi_amt
    df['obi'] = 0.5 * obi_vol + 0.5 * obi_amt
    
    # --- 2c. 盘口深度斜率 (Order Book Slope) ---
    # 买卖盘口的陡峭程度反映供需压力
    bid_slope = np.zeros(len(df))
    ask_slope = np.zeros(len(df))
    for i in range(2, 11):
        bid_slope += (df[f'bid_price{i}'] - df[f'bid_price{i-1}']) / (
            df[f'bid_price{i-1}'] + 1e-8
        )
        ask_slope += (df[f'ask_price{i}'] - df[f'ask_price{i-1}']) / (
            df[f'ask_price{i-1}'] + 1e-8
        )
    df['bid_slope'] = bid_slope / 9
    df['ask_slope'] = ask_slope / 9
    df['slope_diff'] = df['bid_slope'] - df['ask_slope']
    
    # --- 2d. 订单数质量 (Order Count Quality) ---
    bid_orders = np.zeros(len(df))
    ask_orders = np.zeros(len(df))
    for i in range(1, 11):
        w = weights[i - 1]
        bid_orders += w * df[f'bid_num_orders{i}'].fillna(0)
        ask_orders += w * df[f'ask_num_orders{i}'].fillna(0)
    df['bid_order_cnt'] = bid_orders
    df['ask_order_cnt'] = ask_orders
    df['order_imbalance'] = (bid_orders - ask_orders) / (bid_orders + ask_orders + 1)
    
    # --- 2e. 综合盘口指标 ---
    df['bid_ask_vol_ratio'] = df['total_bid_volume'] / (df['total_ask_volume'] + 1)
    df['bid_ask_spread'] = (df['ask_avg_price'] - df['bid_avg_price']) / (df['bid_avg_price'] + 1e-8)
    df['depth_pressure'] = df['bid_avg_price'] / (df['ask_avg_price'] + 1e-8) - 1
    
    # --- 2f. K线形态特征 ---
    df['intraday_ret'] = (df['price'] - df['open']) / (df['open'] + 1e-8)
    df['high_low_range'] = (df['high'] - df['low']) / (df['pre_close'] + 1e-8)
    df['price_position'] = (df['price'] - df['low']) / (df['high'] - df['low'] + 1e-8)
    df['overnight_gap'] = (df['open'] - df['pre_close']) / (df['pre_close'] + 1e-8)
    df['amplitude'] = (df['high'] - df['low']) / (df['open'] + 1e-8)
    
    # --- 2g. 成交活跃度 ---
    df['trade_intensity'] = df['num_trades'] / (df['volume'] + 1)
    df['avg_trade_size'] = df['amount'] / (df['num_trades'] + 1)
    df['volume_momentum'] = df['volume'] / (df.groupby('instrument')['volume'].shift(1) + 1)
    
    # ============================================
    # 3. 日频聚合
    # ============================================
    agg_dict = {
        'obi': ['mean', 'std', 'skew'],
        'obi_vol': ['mean', 'std'],
        'obi_amt': ['mean', 'std'],
        'slope_diff': ['mean', 'std'],
        'order_imbalance': ['mean', 'std'],
        'bid_order_cnt': 'mean',
        'ask_order_cnt': 'mean',
        'bid_ask_vol_ratio': ['mean', 'std'],
        'bid_ask_spread': 'mean',
        'depth_pressure': ['mean', 'std'],
        'intraday_ret': ['mean', 'std', 'last'],
        'high_low_range': ['mean', 'max'],
        'price_position': ['mean', 'last'],
        'overnight_gap': 'first',
        'amplitude': ['mean', 'max'],
        'trade_intensity': ['mean', 'std'],
        'avg_trade_size': ['mean', 'std'],
        'volume_momentum': 'mean',
        'volume': 'sum',
        'amount': 'sum',
        'price': 'last',
        'open': 'first',
        'high': 'max',
        'low': 'min',
        'pre_close': 'first',
        'num_trades': 'sum',
    }
    
    # Filter out columns that do not exist in df
    agg_dict = {k: v for k, v in agg_dict.items() if k in df.columns}
    
    daily = df.groupby(['date', 'instrument']).agg(agg_dict)
    daily.columns = ['_'.join(c).strip('_') for c in daily.columns]
    daily = daily.reset_index()
    
    # ============================================
    # 4. 日频因子合成
    # ============================================
    
    # --- 4a. OBI信号 (含偏度, 偏度反映极端不平衡) ---
    daily['obi_signal'] = daily['obi_mean'] / (daily['obi_std'] + 1e-8)
    daily['obi_skew_signal'] = daily['obi_skew'].fillna(0)
    
    # --- 4b. 盘口斜率信号 ---
    daily['slope_signal'] = daily['slope_diff_mean'] / (daily['slope_diff_std'] + 1e-8)
    
    # --- 4c. 订单不平衡信号 ---
    daily['order_signal'] = daily['order_imbalance_mean'] / (daily['order_imbalance_std'] + 1e-8)
    
    # --- 4d. 日内趋势信号 ---
    daily['intraday_signal'] = daily['intraday_ret_last'] / (daily['intraday_ret_std'] + 1e-8)
    
    # --- 4e. 波动率信号 (低波偏好) ---
    daily['vol_signal'] = -daily['high_low_range_mean']
    
    # --- 4f. 价格位置信号 (收盘接近高位=强) ---
    daily['position_signal'] = daily['price_position_last'] - 0.5
    
    # --- 4g. 成交量信号 ---
    daily['volume_signal'] = daily['trade_intensity_mean'] / (daily['trade_intensity_std'] + 1e-8)
    
    # --- 4h. 隔夜缺口信号 ---
    daily['gap_signal'] = daily['overnight_gap_first']
    
    # --- 4i. 深度压力信号 ---
    daily['depth_signal'] = daily['depth_pressure_mean'] / (daily['depth_pressure_std'] + 1e-8)
    
    # ============================================
    # 5. 获取财务数据
    # ============================================
    try:
        sql_fin = f"""
        SELECT date, instrument,
        total_current_assets, total_assets,
        total_current_liability, total_liability,
        operating_revenue, operating_cost,
        net_profit, operating_profit,
        moneytary_assets, accounts_receivable,
        inventories, fixed_assets
        FROM bigalpha_2026_financial
        WHERE date >= '{start_date}' AND date <= '{end_date}'
        """
        fin = data_source.read(sql_fin)
        
        if len(fin) > 0 and 'total_assets' in fin.columns:
            fin = fin.sort_values(['date', 'instrument'])
            fin = fin.drop_duplicates(['date', 'instrument'], keep='last')
            
            # 计算财务比率
            fin['current_ratio'] = (
                fin['total_current_assets'] / (fin['total_current_liability'] + 1)
            )
            fin['debt_ratio'] = (
                fin['total_liability'] / (fin['total_assets'] + 1)
            )
            fin['profit_margin'] = (
                fin['net_profit'] / (fin['operating_revenue'] + 1)
            )
            fin['asset_turnover'] = (
                fin['operating_revenue'] / (fin['total_assets'] + 1)
            )
            fin['roa'] = fin['net_profit'] / (fin['total_assets'] + 1)
            fin['cash_ratio'] = (
                fin['moneytary_assets'] / (fin['total_current_liability'] + 1)
            )
            
            # 合并
            fin_merge = fin[['date', 'instrument', 'current_ratio', 'debt_ratio',
                            'profit_margin', 'asset_turnover', 'roa', 'cash_ratio']]
            daily = daily.merge(fin_merge, on=['date', 'instrument'], how='left')
            
            # 财务信号 (截面标准化)
            fin_signals = [
                ('current_ratio', 1),   # 高流动比率 = 好
                ('debt_ratio', -1),     # 高负债率 = 差
                ('profit_margin', 1),   # 高利润率 = 好
                ('asset_turnover', 1),  # 高周转率 = 好
                ('roa', 1),             # 高ROA = 好
                ('cash_ratio', 1),      # 高现金比率 = 好
            ]
            daily['fin_signal'] = 0
            count = 0
            for col, sign in fin_signals:
                if col in daily.columns:
                    z = daily.groupby('date')[col].transform(
                        lambda x: (x - x.median()) / (x.std() + 1e-8)
                    )
                    daily['fin_signal'] += sign * z.fillna(0)
                    count += 1
            if count > 0:
                daily['fin_signal'] /= count
        else:
            daily['fin_signal'] = 0
    except Exception:
        daily['fin_signal'] = 0
    
    # ============================================
    # 6. 因子加权组合
    # ============================================
    signal_weights = [
        ('obi_signal', 0.25),
        ('obi_skew_signal', 0.05),
        ('slope_signal', 0.10),
        ('order_signal', 0.10),
        ('depth_signal', 0.05),
        ('intraday_signal', 0.08),
        ('vol_signal', 0.05),
        ('position_signal', 0.08),
        ('volume_signal', 0.05),
        ('gap_signal', 0.04),
        ('fin_signal', 0.15),
    ]
    
    # 截面标准化每个信号
    for col, _ in signal_weights:
        if col in daily.columns:
            z_col = col + "_z"
            daily[z_col] = daily.groupby('date')[col].transform(
                lambda x: (x - x.mean()) / (x.std() + 1e-8)
            )
    
    # 加权合成
    daily['factor'] = 0
    for col, w in signal_weights:
        z_col = col + "_z"
        if z_col in daily.columns:
            daily['factor'] += w * daily[z_col].fillna(0)
    
    # 去极值 (Winsorize 1%/99%)
    daily['factor'] = daily.groupby('date')['factor'].transform(
        lambda x: x.clip(x.quantile(0.01), x.quantile(0.99))
    )
    
    # 截面标准化
    daily['factor'] = daily.groupby('date')['factor'].transform(
        lambda x: (x - x.mean()) / (x.std() + 1e-8)
    )
    
    # 时序平滑 (5日滚动平均)
    daily = daily.sort_values(['instrument', 'date'])
    daily['factor'] = daily.groupby('instrument')['factor'].transform(
        lambda x: x.rolling(5, min_periods=1).mean()
    )
    
    result = daily[['date', 'instrument', 'factor']].copy()
    result['date'] = pd.to_datetime(result['date'])
    return result